# News Modeling

Topic modeling involves **extracting features from document terms** and using
mathematical structures and frameworks like matrix factorization and SVD to generate **clusters or groups of terms** that are distinguishable from each other and these clusters of words form topics or concepts

Topic modeling is a method for **unsupervised classification** of documents, similar to clustering on numeric data

These concepts can be used to interpret the main **themes** of a corpus and also make **semantic connections among words that co-occur together** frequently in various documents

Topic modeling can help in the following areas:
- discovering the **hidden themes** in the collection
- **classifying** the documents into the discovered themes
- using the classification to **organize/summarize/search** the documents

Frameworks and algorithms to build topic models:
- Latent semantic indexing
- Latent Dirichlet allocation
- Non-negative matrix factorization

## Latent Dirichlet Allocation (LDA)
The latent Dirichlet allocation (LDA) technique is a **generative probabilistic model** where each **document is assumed to have a combination of topics** similar to a probabilistic latent semantic indexing model

In simple words, the idea behind LDA is that of two folds:
- each **document** can be described by a **distribution of topics**
- each **topic** can be described by a **distribution of words**

### LDA Algorithm

- 1. For each document, **randomly initialize each word to one of the K topics** (k is chosen beforehand)
- 2. For each document D, go through each word w and compute:
    - **P(T |D)** , which is a proportion of words in D assigned to topic T
    - **P(W |T )** , which is a proportion of assignments to topic T over all documents having the word W
- **Reassign word W with topic T** with probability P(T |D)´ P(W |T ) considering all other words and their topic assignments

![LDA](https://raw.githubusercontent.com/subashgandyer/datasets/main/images/LDA.png)

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Stop words removal
    - Email removal
    - Non-alphabetic words removal
    - Tokenize
    - Lowercase
    - BiGrams & TriGrams
    - Lemmatization
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Evaluate the Topic Model
    - Model Perplexity
    - Topic Coherence
- Visualize the topics

### Install the necessary library

In [1]:
#! pip install pyLDAvis gensim spacy

### Import the libraries

In [13]:
from pprint import pprint

import pandas as pd

import gensim
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.corpora import Dictionary

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer



### Download the dataset
Dataset: https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

#### 20-Newsgroups dataset
- 11K newsgroups posts
- 20 news topics

### Load the dataset

In [14]:
df = pd.read_json("newsgroups.json")
df.head()

,content,target,target_names
0,From: lerxst@wam.umd.edu (where's my thing)\nS...,7,rec.autos
1,From: guykuo@carson.u.washington.edu (Guy Kuo)...,4,comp.sys.mac.hardware
2,From: twillis@ec.ecn.purdue.edu (Thomas E Will...,4,comp.sys.mac.hardware
3,From: jgreen@amber (Joe Green)\nSubject: Re: W...,1,comp.graphics
4,From: jcm@head-cfa.harvard.edu (Jonathan McDow...,14,sci.space


In [15]:
df.columns


Index(['content', 'target', 'target_names'], dtype='object')

### Preprocess the data

### Email Removal

In [16]:
df['content'] = df['content'].str.replace(r'\S+@\S+', '', regex=True)

In [17]:
df.head()

,content,target,target_names
0,From: (where's my thing)\nSubject: WHAT car i...,7,rec.autos
1,From: (Guy Kuo)\nSubject: SI Clock Poll - Fin...,4,comp.sys.mac.hardware
2,From: (Thomas E Willis)\nSubject: PB question...,4,comp.sys.mac.hardware
3,From: (Joe Green)\nSubject: Re: Weitek P9000 ...,1,comp.graphics
4,From: (Jonathan McDowell)\nSubject: Re: Shutt...,14,sci.space


### Newline Removal

In [18]:
df['content'] = df['content'].str.replace('\n', ' ')
df['content'] = df['content'].str.replace(r'\s+', ' ', regex=True)

### Single Quotes Removal

In [19]:
df['content'] = df['content'].str.replace("'", "")

### Tokenize
- Create **sent_to_words()** 
    - Use **gensim.utils.simple_preprocess**
    - Use **generator** instead of an usual function

In [20]:
from gensim.utils import simple_preprocess

def sent_to_words(sentences):
    for sentence in sentences:
        yield simple_preprocess(str(sentence), deacc=True)

data_words = list(sent_to_words(df['content']))

In [21]:
data_words[1]

['from',
 'guy',
 'kuo',
 'subject',
 'si',
 'clock',
 'poll',
 'final',
 'call',
 'summary',
 'final',
 'call',
 'for',
 'si',
 'clock',
 'reports',
 'keywords',
 'si',
 'acceleration',
 'clock',
 'upgrade',
 'article',
 'shelley',
 'qvfo',
 'innc',
 'organization',
 'university',
 'of',
 'washington',
 'lines',
 'nntp',
 'posting',
 'host',
 'carson',
 'washington',
 'edu',
 'fair',
 'number',
 'of',
 'brave',
 'souls',
 'who',
 'upgraded',
 'their',
 'si',
 'clock',
 'oscillator',
 'have',
 'shared',
 'their',
 'experiences',
 'for',
 'this',
 'poll',
 'please',
 'send',
 'brief',
 'message',
 'detailing',
 'your',
 'experiences',
 'with',
 'the',
 'procedure',
 'top',
 'speed',
 'attained',
 'cpu',
 'rated',
 'speed',
 'add',
 'on',
 'cards',
 'and',
 'adapters',
 'heat',
 'sinks',
 'hour',
 'of',
 'usage',
 'per',
 'day',
 'floppy',
 'disk',
 'functionality',
 'with',
 'and',
 'floppies',
 'are',
 'especially',
 'requested',
 'will',
 'be',
 'summarizing',
 'in',
 'the',
 'next',


### Stop words Removal
- Extend the stop words corpus with the following words
    - from
    - subject
    - re
    - edu
    - use

In [22]:
from nltk.corpus import stopwords

stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

#### remove_stopwords( )

In [23]:
def remove_stopwords(texts):
    return [[word for word in doc if word not in stop_words] for doc in texts]

In [24]:
data_words_nostops = remove_stopwords(data_words)

In [25]:
data_words_nostops[1]

['guy',
 'kuo',
 'si',
 'clock',
 'poll',
 'final',
 'call',
 'summary',
 'final',
 'call',
 'si',
 'clock',
 'reports',
 'keywords',
 'si',
 'acceleration',
 'clock',
 'upgrade',
 'article',
 'shelley',
 'qvfo',
 'innc',
 'organization',
 'university',
 'washington',
 'lines',
 'nntp',
 'posting',
 'host',
 'carson',
 'washington',
 'fair',
 'number',
 'brave',
 'souls',
 'upgraded',
 'si',
 'clock',
 'oscillator',
 'shared',
 'experiences',
 'poll',
 'please',
 'send',
 'brief',
 'message',
 'detailing',
 'experiences',
 'procedure',
 'top',
 'speed',
 'attained',
 'cpu',
 'rated',
 'speed',
 'add',
 'cards',
 'adapters',
 'heat',
 'sinks',
 'hour',
 'usage',
 'per',
 'day',
 'floppy',
 'disk',
 'functionality',
 'floppies',
 'especially',
 'requested',
 'summarizing',
 'next',
 'two',
 'days',
 'please',
 'add',
 'network',
 'knowledge',
 'base',
 'done',
 'clock',
 'upgrade',
 'havent',
 'answered',
 'poll',
 'thanks',
 'guy',
 'kuo']

### Bigrams
- Use **gensim.models.Phrases**
- 100 as threshold

In [29]:
from gensim.models import Phrases
from gensim.models.phrases import Phraser

bigram = Phrases(data_words_nostops, min_count=5, threshold=100)
bigram_mod = Phraser(bigram)

#### make_bigrams( )

In [30]:
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

In [32]:
data_words_bigrams = make_bigrams(data_words_nostops)

In [33]:
data_words_bigrams[2]


['thomas',
 'willis',
 'pb',
 'questions',
 'organization',
 'purdue_university',
 'engineering',
 'computer',
 'network',
 'distribution_usa',
 'lines',
 'well',
 'folks',
 'mac',
 'plus',
 'finally',
 'gave',
 'ghost',
 'weekend',
 'starting',
 'life',
 'way',
 'back',
 'sooo',
 'im',
 'market',
 'new',
 'machine',
 'bit',
 'sooner',
 'intended',
 'im',
 'looking',
 'picking',
 'powerbook',
 'maybe',
 'bunch',
 'questions',
 'hopefully',
 'somebody',
 'answer',
 'anybody',
 'know',
 'dirt',
 'next',
 'round',
 'powerbook',
 'introductions',
 'expected',
 'id',
 'heard',
 'supposed',
 'make',
 'appearence',
 'summer',
 'havent',
 'heard',
 'anymore',
 'since',
 'dont',
 'access',
 'macleak',
 'wondering',
 'anybody',
 'info',
 'anybody',
 'heard_rumors',
 'price',
 'drops',
 'powerbook',
 'line',
 'like',
 'ones',
 'duos',
 'went',
 'recently',
 'whats',
 'impression',
 'display',
 'could',
 'probably',
 'swing',
 'got',
 'mb',
 'disk',
 'rather',
 'dont',
 'really',
 'feel',
 'much',

### Lemmatization
- Use spacy
    - Download spacy en model (if you have not done that before)
    - Load the spacy model

In [27]:
#! python -m spacy download en

In [35]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

#### lemmatizaton( )

In [36]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [37]:
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [39]:
print(data_lemmatized[:1])

[['s', 'thing', 'car', 'nntp_poste', 'host', 'park', 'line', 'wonder', 'enlighten', 'car', 'see', 'day', 'door', 'sport', 'car', 'look', 'late', 'early', 'call', 'bricklin', 'door', 'really', 'small', 'addition', 'separate', 'rest', 'body', 'know', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'make', 'history', 'info', 'funky', 'look', 'car', 'mail', 'thank', 'bring', 'neighborhood', 'lerxst']]


### Create a Dictionary

In [44]:
dictionary = Dictionary(data_lemmatized)
dictionary[1]

'body'

In [45]:
dictionary.token2id["group"]

874

### Filter low-frequency words

In [52]:
dictionary.filter_extremes(no_below=5, no_above=0.5)

### Create Corpus

In [53]:
corpus = [dictionary.doc2bow(text) for text in data_lemmatized]

### Create Index 2 word dictionary

In [54]:
temp = dictionary[0]  # This is only to "load" the dictionary.


In [55]:
id2word = dictionary.id2token

### Build a News Topic Model

#### LdaModel
- **num_topics** : this is the number of topics you need to define beforehand
- **chunksize** : the number of documents to be used in each training chunk
- **alpha** : this is the hyperparameters that affect the sparsity of the topics
- **passess** : total number of training assess

In [60]:
ldamodel = LdaModel(corpus, num_topics=10, chunksize=100, id2word = id2word, alpha='auto', passes=10, random_state=42)

### Print the Keyword in the 10 topics

In [61]:
pprint(ldamodel.top_topics(corpus,topn=10))

[([(np.float32(0.026194653), 'get'),
   (np.float32(0.021725284), 'article'),
   (np.float32(0.018846756), 'go'),
   (np.float32(0.017788706), 'know'),
   (np.float32(0.016584484), 'nntp_poste'),
   (np.float32(0.015257499), 'm'),
   (np.float32(0.014414245), 'good'),
   (np.float32(0.013282816), 'think'),
   (np.float32(0.01309438), 'host'),
   (np.float32(0.01287107), 'make')],
  np.float64(-1.0269724101377857)),
 ([(np.float32(0.016187696), 'say'),
   (np.float32(0.015051358), 'people'),
   (np.float32(0.010000534), 'believe'),
   (np.float32(0.00922111), 'reason'),
   (np.float32(0.00872809), 'evidence'),
   (np.float32(0.008584038), 'make'),
   (np.float32(0.0074896286), 'many'),
   (np.float32(0.0074573923), 'think'),
   (np.float32(0.006921813), 'point'),
   (np.float32(0.0067925635), 'mean')],
  np.float64(-1.2243650285270349)),
 ([(np.float32(0.01762532), 'drive'),
   (np.float32(0.01682771), 'file'),
   (np.float32(0.015380368), 'window'),
   (np.float32(0.013841594), 'use'),

In [62]:
for idx, topic in ldamodel.print_topics(num_words=10):
    print(f"Topic #{idx}: {topic}")

Topic #0: 0.026*"space" + 0.014*"research" + 0.012*"encryption" + 0.009*"program" + 0.009*"system" + 0.008*"item" + 0.008*"encrypt" + 0.008*"sphere" + 0.008*"launch" + 0.008*"datum"
Topic #1: 0.021*"key" + 0.019*"mail" + 0.015*"use" + 0.015*"information" + 0.014*"system" + 0.013*"send" + 0.013*"new" + 0.012*"chip" + 0.012*"computer" + 0.011*"need"
Topic #2: 0.026*"get" + 0.022*"article" + 0.019*"go" + 0.018*"know" + 0.017*"nntp_poste" + 0.015*"m" + 0.014*"good" + 0.013*"think" + 0.013*"host" + 0.013*"make"
Topic #3: 0.016*"say" + 0.015*"people" + 0.010*"believe" + 0.009*"reason" + 0.009*"evidence" + 0.009*"make" + 0.007*"many" + 0.007*"think" + 0.007*"point" + 0.007*"mean"
Topic #4: 0.024*"year" + 0.023*"team" + 0.021*"game" + 0.014*"play" + 0.013*"win" + 0.009*"last" + 0.009*"lose" + 0.009*"first" + 0.009*"sens" + 0.009*"hockey"
Topic #5: 0.029*"physical" + 0.024*"patient" + 0.014*"face" + 0.012*"eat" + 0.012*"disease" + 0.012*"effect" + 0.011*"cause" + 0.011*"direct" + 0.011*"recomme

## Evaluation of Topic Models
- Model Perplexity
- Topic Coherence

### Model Perplexity

Model perplexity is a measurement of **how well** a **probability distribution** or probability model **predicts a sample**

In [64]:
print('\nPerplexity: ', ldamodel.log_perplexity(corpus))


Perplexity:  -7.97345917438882


### Topic Coherence
Topic Coherence measures score a single topic by measuring the **degree of semantic similarity** between **high scoring words** in the topic.

In [66]:
from gensim.models import CoherenceModel

coherence_model = CoherenceModel(
    model=ldamodel,
    texts=data_lemmatized,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print('\nCoherence Score:', coherence_score)


Coherence Score: 0.5143524033021762


### Model Evaluation

The LDA model was evaluated using both perplexity and coherence score. The model achieved a perplexity score of -7.97, which indicates an improvement in predictive performance compared to the previous configuration.

The coherence score of 0.51 suggests that the topics are semantically meaningful and interpretable. This indicates that the model is able to generate coherent topics with related keywords.

Overall, the model provides a good balance between statistical performance and interpretability, making it suitable for analyzing news topics.

### Visualize the Topic Model
- Use **pyLDAvis**
    - designed to help users **interpret the topics** in a topic model that has been fit to a corpus of text data
    - extracts information from a fitted LDA topic model to inform an interactive web-based visualization

In [67]:
import pyLDAvis.gensim
import warnings
warnings.filterwarnings("ignore")

In [68]:
pyLDAvis.enable_notebook()

In [69]:
pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.222628 -0.105629       1        1  23.999499
3     -0.185976 -0.099243       2        1  23.454065
7     -0.123809  0.209011       3        1  10.533600
1     -0.108732  0.236804       4        1   9.071447
6      0.014651 -0.063040       5        1   7.705680
4     -0.038308 -0.225316       6        1   7.044917
9      0.038004 -0.124422       7        1   5.834721
8      0.330966  0.055978       8        1   4.934156
0      0.029896  0.180749       9        1   4.751106
5      0.265935 -0.064891      10        1   2.670809, topic_info=         Term          Freq         Total Category  logprob  loglift
3652       ax  32991.000000  32991.000000  Default  30.0000  30.0000
106       get   8061.000000   8061.000000  Default  29.0000  29.0000
171   article   6806.000000   6806.000000  Default  28.0000  28.0000
133    people   6033.000000   6033.000000  Default  27.0000  27.0000
109        go   5892.000000   5892.000000  Default  26.0000  26.0000
...       ...           ...           ...      ...      ...      ...
701     study    273.434295    657.283580  Topic10  -4.7243   2.7457
854       age    218.904100    571.491397  Topic10  -4.9467   2.6632
1432     week    234.788945    930.435817  Topic10  -4.8766   2.2458
288    result    231.695436    874.771532  Topic10  -4.8899   2.2942
1        body    210.186497    561.691528  Topic10  -4.9873   2.6398

[526 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
3650      7  0.829787        _
3650      8  0.168602        _
3562      1  0.997103  ability
1238      2  0.998736   accept
170       2  0.107799  address
...     ...       ...      ...
36        5  0.087305     year
36        6  0.529544     year
36        7  0.079690     year
36        9  0.085402     year
36       10  0.018223     year

[841 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 4, 8, 2, 7, 5, 10, 9, 1, 6])

### Interpretation of Topics

The LDA model identified 10 distinct topics from the news dataset, each representing a cluster of semantically related words.

**Topic 0: Space & Technology**
This topic includes words such as *space, research, encryption, launch*, which suggests a combination of space exploration and technical systems, possibly related to aerospace and security technologies.

**Topic 1: Communication & Computer Systems**
Keywords like *mail, system, computer, chip* indicate topics related to digital communication, computing infrastructure, and information exchange.

**Topic 2: General Discussion / Online Posts**
This topic contains common conversational words such as *get, know, think, article*, which suggests general discussion or informal online communication, likely from forums or posts.

**Topic 3: Beliefs & Reasoning**
Words such as *people, believe, evidence, reason* indicate discussions around opinions, arguments, and possibly philosophical or religious debates.

**Topic 4: Sports (Hockey)**
This topic clearly represents sports, with words like *team, game, play, win, hockey*, indicating discussions related to hockey and competitive sports.

**Topic 5: Health & Medicine**
Keywords such as *patient, disease, treatment, physical* suggest medical topics related to healthcare, diagnosis, and treatment.

**Topic 6: Vehicles & Public Safety**
This topic includes *car, gun, fire, report*, which may reflect discussions around transportation, safety, or public policy issues.

**Topic 7: Software & Computer Systems**
Words like *file, window, program, software* clearly indicate topics related to operating systems, programming, and technical issues.

**Topic 8: Noise / Unclear Topic**
This topic is dominated by the term *ax* and contains less coherent words, suggesting noise or poorly clustered data. It may result from preprocessing limitations or irrelevant tokens.

**Topic 9: Politics & Conflict**
Keywords such as *government, state, attack, war, country* indicate discussions related to politics, conflict, and international affairs.

---

### Overall Analysis

The model successfully identified several meaningful and interpretable topics such as sports, health, politics, and technology. Most topics are coherent and semantically meaningful, which aligns with the coherence score obtained earlier.

However, one topic (Topic 8) appears less interpretable, indicating some noise in the data or preprocessing steps. Further cleaning or parameter tuning could improve this.

Overall, the topic model provides a useful high-level overview of the main themes present in the news dataset.
